# Business Understanding

## Context

Platform Ondernemen in Hardenberg ondersteunt ondernemers bij vraagstukken rondom ondernemerschap, organisatieontwikkeling en groei. Binnen deze dienstverlening vormen persoonlijke adviesgesprekken een belangrijk onderdeel van het proces. Tijdens deze gesprekken proberen adviseurs inzicht te krijgen in de situatie van een ondernemer om vervolgens passende begeleiding en ondersteuning te bieden.

Binnen de huidige werkwijze kost het adviesproces relatief veel tijd. De kern van een vraagstuk wordt vaak pas tijdens het gesprek duidelijk, waardoor gesprekken eerst een oriënterend karakter hebben voordat verdieping mogelijk is. Daarnaast vindt verslaglegging en opvolging grotendeels handmatig plaats, wat zorgt voor extra administratieve belasting en minder structuur binnen het proces.

Om de dienstverlening verder te professionaliseren onderzoekt Platform Ondernemen de mogelijkheden van digitalisering en Artificial Intelligence (AI). Hierbij staat centraal dat technologie ondersteunend moet zijn aan de adviseur en het persoonlijke contact niet mag vervangen. Een mogelijke toepassing hiervan is een dynamische intakevragenlijst die ondernemers voorafgaand aan een gesprek helpt om hun situatie en hulpvraag beter in kaart te brengen.

Binnen deze Proof of Concept (PoC) wordt onderzocht welke Large Language Model (LLM) via Ollama het beste aansluit op deze use case. Hiervoor wordt een testomgeving ontwikkeld in Jupyter Notebook waarin verschillende modellen met elkaar worden vergeleken op basis van hun prestaties binnen het intakeproces.

---

## Doelstelling

Het doel van dit Jupyter Notebook is het vergelijken en evalueren van verschillende Large Language Models (LLM’s) binnen Ollama om te bepalen welk model het meest geschikt is voor het ondersteunen van een dynamische intakevragenlijst voor Platform Ondernemen.

Tijdens deze vergelijking wordt onderzocht in hoeverre de modellen:

* relevante en contextgerichte intakevragen genereren;
* logisch kunnen doorvragen op basis van eerdere antwoorden;
* duidelijke en professionele communicatie bieden;
* consistent reageren binnen een gesprekssituatie;
* efficiënt en bruikbaar zijn binnen een toekomstige webapplicatie.

De resultaten van deze analyse moeten inzicht geven in welk LLM-model het beste aansluit bij de behoeften van Platform Ondernemen en het meest geschikt is voor verdere ontwikkeling binnen de PoC.



# Data Understanding
Er is bij dit project geen data geleverd om te verkennen. Bij deze stap in de CRISP-DM cyclus worden hierom enkel de imports en de vragenlijst beschreven.

## Importeren van benodigde libraries

In [87]:
# Imports regelen
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import ollama
import requests
from typing import List, Dict

## Opstellen dynamische vragenlijst

In [88]:
# Flow voor vragenlijst die straks getest moet worden op verschillende LLM's. 
flow = [
    # Handmatige vragen die sowieso gesteld moeten worden
    {"type": "manual", "question": "Wat is je voor- en achternaam?"},
    {"type": "manual", "question": "Wat is de naam van je bedrijf?"},
    {"type": "manual", "question": "Wat doet je bedrijf precies?"},

    # Thema 1: Huidige situatie van het bedrijf
    {"type": "ai", "theme": "Huidige situatie van het bedrijf"},
    {"type": "ai", "theme": "Huidige situatie van het bedrijf"},
    {"type": "ai", "theme": "Huidige situatie van het bedrijf"},

    # Thema 2: Tijdsbesteding en prioriteiten
    {"type": "ai", "theme": "Tijdsbesteding en prioriteiten"},
    {"type": "ai", "theme": "Tijdsbesteding en prioriteiten"},
    {"type": "ai", "theme": "Tijdsbesteding en prioriteiten"},

    # Thema 3: Organisatie en samenwerking
    {"type": "ai", "theme": "Organisatie en samenwerking"},
    {"type": "ai", "theme": "Organisatie en samenwerking"},
    {"type": "ai", "theme": "Organisatie en samenwerking"},

    # Thema 4: Uitdagingen en ontwikkelpunten
    {"type": "ai", "theme": "Uitdagingen en ontwikkelpunten"},
    {"type": "ai", "theme": "Uitdagingen en ontwikkelpunten"},
    {"type": "ai", "theme": "Uitdagingen en ontwikkelpunten"},

    # Thema 5: Toekomst en groeikansen
    {"type": "ai", "theme": "Toekomst en groeikansen"},
    {"type": "ai", "theme": "Toekomst en groeikansen"},
    {"type": "ai", "theme": "Toekomst en groeikansen"},

    # Thema 6: Verwachting van het gesprek
    {"type": "ai", "theme": "Verwachting van het gesprek"},
    {"type": "ai", "theme": "Verwachting van het gesprek"},
    {"type": "ai", "theme": "Verwachting van het gesprek"}
]

In [89]:
# Lijst met verschillende LLM's die getest moeten worden
llms = [
    "gemma3:4b"
]

## Data Evaluation
Voor de dynamische intakevragenlijst is vooraf bepaald welke informatie nodig is voor adviseurs van Platform Ondernemen om goed voorbereid een gesprek in te gaan. De onderwerpen en thema’s zijn vastgesteld door de opdrachtgever, op basis van wat zij belangrijk vinden in een intakegesprek.

De intake is opgebouwd uit zes vaste thema’s:

1. Huidige situatie van het bedrijf
2. Tijdsbesteding en prioriteiten
3. Organisatie en samenwerking
4. Uitdagingen en ontwikkelpunten
5. Toekomst en groeikansen
6. Verwachting van het gesprek

Daarnaast zijn er drie standaardvragen toegevoegd om basisinformatie van de ondernemer en het bedrijf te verzamelen.

De opdrachtgever heeft hierbij aangegeven dat de intake:

* Makkelijk en snel in te vullen moet zijn.
* Ongeveer 3 tot 5 minuten mag duren.
* Direct moet helpen om de kern van het probleem te vinden.
* Genoeg diepgang moet geven om het gesprek goed voor te bereiden.
* Persoonlijk en laagdrempelig moet blijven.

Binnen het Jupyter Notebook is de dynamische vragenlijst opgezet tijdens de Data Understanding. Eerst worden de benodigde imports ingeladen om met de AI (Ollama) te kunnen werken. Daarna wordt een vaste flow opgezet waarin per stap wordt bepaald of een vraag handmatig wordt gesteld of door een LLM wordt gegenereerd. Ook zijn de verschillende LLM's die getest zullen worden hiering gedefinieerd. 

De zes thema’s worden gebruikt als context voor het model. Op basis hiervan kan het LLM gerichte vervolgvragen stellen die passen bij de antwoorden van de ondernemer. Zo blijft de intake flexibel, maar wordt er toch gericht doorgevraagd zonder dat het proces te lang of ingewikkeld wordt.

# Data Preparation

## Voorbereiden systemprompt

In [90]:
def build_system_prompt(theme):
    return f"""
Je bent een zakelijke intake assistent.

Je taak:
- Stel EXACT 1 korte zakelijke vraag
- De vraag moet passen binnen het thema: "{theme}"

REGELS:
- Alleen een vraagzin
- Geen uitleg
- Geen intro
- Geen opsommingen
- Geen labels
- Geen meerdere vragen
- Geen herhaling
- Maximaal 1 zin

VOORBEELDEN:
"Wat kost momenteel de meeste tijd binnen jullie bedrijf?"
"Waar lopen medewerkers het vaakst tegenaan?"
"Welke processen verlopen nog handmatig?"

Output uitsluitend de vraag.
"""

In [91]:
ANSWER_SYSTEM_PROMPT = """
Je bent de eigenaar/medewerker van een bedrijf dat een intake gesprek voert met een adviseur.

Je antwoordt altijd realistisch, kort en zakelijk.

REGELS:
- Antwoord alsof je in een echt bedrijf werkt
- Geen AI-stijl of uitleg
- Maximaal 1-2 zinnen
- Nooit herhalen van de vraag
- Geef concrete bedrijfsinformatie (tijd, problemen, processen, prioriteiten)
- Als informatie ontbreekt: maak een realistische aanname passend bij een MKB-bedrijf

STIJL:
- Natuurlijk Nederlands
- Zakelijk maar menselijk
- Niet formeel overdreven
"""

In [92]:
def build_history(flow_history, current_theme=None):
    """
    flow_history: list van dicts met rol + content
    """
    history = []

    for h in flow_history:
        # alleen relevante context meenemen
        if h["type"] == "manual":
            history.append({
                "role": "user",
                "content": f"{h['question']} -> {h.get('answer', '')}"
            })

        if h["type"] == "ai" and current_theme and h.get("theme") == current_theme:
            history.append({
                "role": "assistant",
                "content": h.get("question", "")
            })

    return history

# Modeling

In [93]:
def call_llm(model: str, system_prompt: str, messages: list) -> str:
    url = "http://localhost:11434/api/generate"

    # combine system + messages tot 1 prompt (belangrijk bij /generate)
    prompt = system_prompt + "\n\n"

    for m in messages:
        prompt += f"{m['role']}: {m['content']}\n"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()

    return response.json()["response"]

In [94]:
def generate_answer(model: str, question: str):
    messages = [
        {
            "role": "user",
            "content": f"Beantwoord deze vraag:\n\n{question}"
        }
    ]

    return call_llm(model, ANSWER_SYSTEM_PROMPT, messages)

In [95]:
def generate_question(model: str, theme: str, theme_history: str):
    system = build_system_prompt(theme)

    # EXACT hetzelfde als frontend server message
    user_message = theme_history or ""

    messages = [
        {"role": "user", "content": user_message}
    ]

    return call_llm(model, system, messages)

In [96]:
def run_flow(flow, llms):

    theme_histories = {}
    manual_history = {}
    results = []

    for model in llms:

        for f in flow:

            theme = f.get("theme")

            # 1. MANUAL
            if f["type"] == "manual":
                question = f["question"]

                answer = generate_answer(model, question)

                manual_history["manual"] = manual_history.get("manual", "")
                manual_history["manual"] += f"Vraag: {question}\nAntwoord: {answer}\n\n"

            # 2. AI
            else:
                manual = manual_history.get("manual", "")
                theme_history = theme_histories.get(theme, "")

                history = f"""
                --- ALGEMENE CONTEXT (MANUAL) ---
                {manual}

                --- THEMA CONTEXT ---
                {theme_history}
                """

                question = generate_question(model, theme, history)

                answer = generate_answer(model, question)

                # update theme memory
                theme_histories[theme] = theme_histories.get(theme, "")
                theme_histories[theme] += f"Vraag: {question}\n"
                theme_histories[theme] += f"Antwoord: {answer}\n\n"

            results.append({
                "model": model,
                "type": f["type"],
                "theme": theme,
                "question": question,
                "answer": answer
            })

    return pd.DataFrame(results)

# Evaluation

In [97]:
df = run_flow(flow, llms)

print(df.head())

       model    type                             theme  \
0  gemma3:4b  manual                              None   
1  gemma3:4b  manual                              None   
2  gemma3:4b  manual                              None   
3  gemma3:4b      ai  Huidige situatie van het bedrijf   
4  gemma3:4b      ai  Huidige situatie van het bedrijf   

                                            question  \
0                     Wat is je voor- en achternaam?   
1                     Wat is de naam van je bedrijf?   
2                       Wat doet je bedrijf precies?   
3  Hoeveel tijd besteden jullie momenteel aan het...   
4  Wat zijn de grootste knelpunten in jullie huid...   

                                              answer  
0  Mijn naam is Pieter Jansen. We hebben momentee...  
1  We heten ‘Innovatie Partners’. We zijn gespeci...  
2  Wij zijn Innovatie Partners, gespecialiseerd i...  
3  Momenteel besteden we gemiddeld 4-6 uur per we...  
4  We ervaren momenteel vertragingen in

In [98]:
df.head()

,model,type,theme,question,answer
0,gemma3:4b,manual,None,Wat is je voor- en achternaam?,Mijn naam is Pieter Jansen. We hebben momentee...
1,gemma3:4b,manual,None,Wat is de naam van je bedrijf?,We heten ‘Innovatie Partners’. We zijn gespeci...
2,gemma3:4b,manual,None,Wat doet je bedrijf precies?,"Wij zijn Innovatie Partners, gespecialiseerd i..."
3,gemma3:4b,ai,Huidige situatie van het bedrijf,Hoeveel tijd besteden jullie momenteel aan het...,Momenteel besteden we gemiddeld 4-6 uur per we...
4,gemma3:4b,ai,Huidige situatie van het bedrijf,Wat zijn de grootste knelpunten in jullie huid...,We ervaren momenteel vertragingen in de orderv...


In [99]:
print(df.to_string(index=False))

    model   type                            theme                                                                                                question                                                                                                                                                                                                                                                         answer
gemma3:4b manual                             None                                                                          Wat is je voor- en achternaam?                                                                                                                                                     Mijn naam is Pieter Jansen. We hebben momenteel prioriteit aan de doorlooptijd van onze offerte-aanvragen.
gemma3:4b manual                             None                                                                          Wat is de naam van je bedrijf?                             